# Clona tu voz y haz que lea por ti
### Tutorial con Chatterbox Multilingual, un modelo de código abierto

Grabas quince segundos de tu voz, escribes un texto y un modelo lo lee **con tu voz**, con la entonación que tú le pidas. Todo ocurre en tu ordenador: ni tu grabación ni el texto salen de él.

| # | Paso | Qué haces |
|---|---|---|
| 1 | Preparar el entorno | Un entorno de Python nuevo, con el modelo |
| 2 | Cargar el modelo | Chatterbox Multilingual, 23 idiomas |
| 3 | Grabar tu voz | Quince segundos con el micrófono, y comprobar que la grabación vale |
| 4 | La primera frase con tu voz | Texto escrito → audio |
| 5 | Hacerla expresiva | Los tres mandos del modelo, y cómo se escribe un texto para ser leído |
| 6 | Un texto largo, con guion | Cada frase con su tono, unidas en una sola lectura |
| 7 | Tu voz en otro idioma | La misma voz hablando inglés, francés o japonés |
| 8 | ¿Se parece? | Medir el parecido en vez de fiarse del oído |
| 9 | Marca de agua y uso responsable | Cómo se detecta que un audio es sintético, y qué no se hace con esto |

## Cómo funciona, en dos frases

Los modelos de hace unos años necesitaban horas de grabaciones y un entrenamiento por cada voz. Los actuales, como este, **no se entrenan con tu voz**: reciben unos segundos de audio como *referencia*, extraen de ahí el timbre y la forma de hablar, y generan el audio nuevo imitándolos. Por eso se llama clonación *zero-shot*: sin ningún ejemplo de entrenamiento.

## Una regla antes de empezar

**Clona solo tu propia voz, o la de alguien que te haya dado permiso.** La voz identifica a una persona igual que su cara. En el paso 9 volvemos sobre esto.

---
# 1 · Preparar el entorno

Chatterbox necesita versiones concretas de varias librerías (entre ellas PyTorch 2.6). Si lo instalas en un entorno que ya usas para otra cosa, te las cambiará. Por eso se instala en **un entorno nuevo**, solo para este tutorial.

Con **Python 3.11**, en una terminal abierta en la carpeta de este cuaderno:

```bash
python3.11 -m venv .venv-voz
source .venv-voz/bin/activate            # en Windows: .venv-voz\Scripts\activate
pip install -r requirements.txt
```

Después abre este cuaderno y elige como kernel el Python de `.venv-voz`.

El archivo `requirements.txt` contiene esto:

| Paquete | Para qué |
|---|---|
| `chatterbox-tts` | El modelo |
| `sounddevice` | Grabar con el micrófono desde el cuaderno |
| `matplotlib` | Dibujar la onda de tu grabación |
| `setuptools<81` | Lo necesita una pieza de Chatterbox (la marca de agua). Sin él, el modelo falla al cargar con el error `'NoneType' object is not callable` |
| `ipykernel` | Para usar el entorno desde el cuaderno |

Dos cosas más:

- **La primera vez se descarga el modelo: unos 3 GB.** Después queda guardado.
- **No hace falta tarjeta gráfica.** Con GPU, o en un Mac con chip de Apple, una frase tarda unos segundos. Sin ella funciona igual, pero cada segundo de audio cuesta varios segundos de espera.

In [ ]:
!pip install -r /Users/deme/Desktop/Quorax/clonar-voz-chatterbox/requirements.txt

In [ ]:
import logging
import re
import time
import warnings
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, Markdown, display

warnings.filterwarnings("ignore")
logging.getLogger("chatterbox").setLevel(logging.ERROR)        # sin los avisos internos del modelo
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

SALIDAS = Path("audios")                 # aquí se guarda todo lo que generemos
SALIDAS.mkdir(exist_ok=True)

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("PyTorch", torch.__version__, "· se va a usar:", DISPOSITIVO)

---
# 2 · Cargar el modelo

Chatterbox Multilingual lo publica Resemble AI con licencia MIT: se puede usar, modificar y llevar a un producto. Tiene unos 500 millones de parámetros, un tamaño que cabe en un portátil.

Por dentro son tres piezas, y conviene saber que existen porque explican lo que vamos a ver:

| Pieza | Qué hace |
|---|---|
| Codificador de voz | Convierte tu grabación en un vector que resume **quién habla**: el timbre |
| Modelo de lenguaje | Lee el texto y decide **cómo** decirlo: ritmo, pausas, entonación |
| Decodificador | Convierte esa decisión en la onda de sonido, con el timbre de la referencia |

In [ ]:
from chatterbox.mtl_tts import SUPPORTED_LANGUAGES, ChatterboxMultilingualTTS

inicio = time.time()
modelo = ChatterboxMultilingualTTS.from_pretrained(device=DISPOSITIVO)
VOZ_DE_FABRICA = modelo.conds          # la voz que trae el modelo; la usaremos en el paso 8 para comparar

print(f"Modelo cargado en {time.time() - inicio:.0f} s · genera audio a {modelo.sr} Hz")
print(len(SUPPORTED_LANGUAGES), "idiomas:", ", ".join(f"{codigo} ({nombre})" for codigo, nombre in SUPPORTED_LANGUAGES.items()))

---
# 3 · Grabar tu voz

El modelo imita lo que oye. Si la grabación tiene eco, ruido o música, la voz clonada los tendrá. Si lees como un robot, hablará como un robot.

## Cómo grabar bien

- **Entre 10 y 20 segundos.** Más no mejora el resultado; menos de 8 se queda corto.
- **Un sitio silencioso y sin eco.** Una habitación con cortinas, sofá o ropa es mejor que una cocina o un pasillo.
- **El micrófono a un palmo de la boca**, y siempre a la misma distancia.
- **Habla como quieres que suene la copia.** Con naturalidad, con algo de entonación, sin prisa.
- **En el mismo idioma en que va a hablar.** Si la referencia está en español, el español sale mejor.

## Qué decir

Cualquier cosa sirve, pero es mejor un texto con frases variadas, alguna pregunta y alguna exclamación. Puedes leer este, que dura unos quince segundos:

> *Hola, esta es mi voz. Te cuento lo que me pasó ayer por la tarde: salí a dar un paseo, empezó a llover de repente y acabé merendando chocolate caliente en una cafetería pequeña. ¿Y sabes qué? Fue el mejor rato de toda la semana.*

## Grabar

La celda siguiente graba con el micrófono del ordenador. Hace una cuenta atrás, graba `SEGUNDOS` segundos y guarda el resultado en `mi_voz.wav`.

- La primera vez, el sistema te pedirá permiso para que el programa donde tienes abierto el cuaderno use el micrófono.
- Si ya existe `mi_voz.wav`, la celda no graba encima. Para repetir la grabación, pon `REGRABAR = True`.
- Si prefieres grabarte con el móvil o con otro programa, guarda el archivo junto al cuaderno y escribe su nombre en `IMPORTAR`. Valen `.wav` y `.mp3`.

In [ ]:
ARCHIVO_DE_VOZ = Path("mi_voz.wav")
SEGUNDOS = 15
REGRABAR = False
IMPORTAR = None                # por ejemplo "grabacion_del_movil.mp3"; None para grabar aquí

if IMPORTAR:
    audio, frecuencia = librosa.load(IMPORTAR, sr=None, mono=True)
    sf.write(ARCHIVO_DE_VOZ, audio, frecuencia)
    print(f"Importado {IMPORTAR} → {ARCHIVO_DE_VOZ}")

elif ARCHIVO_DE_VOZ.exists() and not REGRABAR:
    print(f"Ya existe {ARCHIVO_DE_VOZ}: se usa esa grabación. Para repetirla, pon REGRABAR = True.")

else:
    import sounddevice as sd

    microfono = sd.query_devices(kind="input")
    frecuencia = int(microfono["default_samplerate"])
    print("Micrófono:", microfono["name"])

    for cuenta in ["3", "2", "1"]:
        print(cuenta, end=" … ", flush=True)
        time.sleep(1)
    print("¡HABLA!")

    grabacion = sd.rec(int(SEGUNDOS * frecuencia), samplerate=frecuencia, channels=2, dtype="float32")
    for quedan in range(SEGUNDOS, 0, -1):
        print(quedan, end=" ", flush=True)
        time.sleep(1)
    sd.wait()

    sf.write(ARCHIVO_DE_VOZ, grabacion[:, 0], frecuencia)
    print(f"\nGuardado en {ARCHIVO_DE_VOZ}")

## ¿Vale la grabación?

Antes de clonar nada, se escucha y se mira. Tres comprobaciones, con sus números:

| Qué | Por qué importa |
|---|---|
| **Duración** | Con menos de 8 segundos el modelo tiene poco de donde sacar el timbre |
| **Volumen** | Si está muy baja, el ruido de fondo pesa demasiado |
| **Saturación** | Si la onda toca el techo, el sonido está distorsionado y la copia hereda la distorsión |

In [ ]:
mi_voz, frecuencia = librosa.load(ARCHIVO_DE_VOZ, sr=None, mono=True)
duracion = len(mi_voz) / frecuencia
pico = float(np.abs(mi_voz).max())
volumen = float(np.sqrt((mi_voz ** 2).mean()))

fig, eje = plt.subplots(figsize=(9, 2.2))
eje.plot(np.arange(len(mi_voz)) / frecuencia, mi_voz, linewidth=0.4, color="#0147FF")
eje.set_ylim(-1, 1)
eje.set_xlabel("segundos")
eje.set_title("Tu grabación", loc="left", fontweight="bold")
eje.spines[["top", "right"]].set_visible(False)
plt.show()

display(Audio(mi_voz, rate=frecuencia))


def revisar(que, bien, consejo):
    print(f"✔ {que}" if bien else f"✘ {que}\n   {consejo}")


revisar(f"Duración: {duracion:.1f} s", duracion >= 8, "Graba al menos 10 segundos.")
revisar(f"Volumen medio: {volumen:.3f}", volumen >= 0.01, "Se oye muy bajo: acércate al micrófono o sube la entrada en los ajustes de sonido.")
revisar(f"Pico máximo: {pico:.2f}", pico < 0.99, "La grabación satura: aléjate un poco o baja la entrada del micrófono.")

Escúchala con auriculares. Si oyes eco, un ventilador o un golpe en la mesa, repítela: son dos minutos y la diferencia en el resultado es grande.

---
# 4 · La primera frase con tu voz

Todo el trabajo lo hace una función, `modelo.generate`. Recibe el texto, el idioma y la ruta de tu grabación, y devuelve la onda de sonido.

Para no repetir lo mismo en cada celda, la envolvemos en dos funciones nuestras:

- `hablar(texto)` genera el audio y lo devuelve.
- `escuchar(audio)` lo reproduce en el cuaderno.

In [ ]:
ESTILOS = {
    "plano":     dict(exaggeration=0.25, cfg_weight=0.5),
    "natural":   dict(exaggeration=0.5, cfg_weight=0.5),      # los valores por defecto del modelo
    "expresivo": dict(exaggeration=0.7, cfg_weight=0.3),
    "teatral":   dict(exaggeration=0.9, cfg_weight=0.3),
}


def hablar(texto, estilo="natural", idioma="es", semilla=7, guardar=None, intentos=3, **ajustes):
    """Devuelve el audio del texto leído con tu voz. Con `guardar`, además lo escribe en la carpeta de audios."""
    opciones = {**ESTILOS[estilo], **ajustes}
    limite = 2.0 + 0.075 * len(texto)         # segundos que, como mucho, debería durar este texto
    for intento in range(intentos):
        torch.manual_seed(semilla + intento)
        onda = modelo.generate(texto, language_id=idioma, audio_prompt_path=str(ARCHIVO_DE_VOZ), **opciones)
        audio = onda.squeeze(0).cpu().numpy()
        if len(audio) / modelo.sr <= limite or idioma in {"zh", "ja", "ko"}:
            break
        print(f"   ↻ «{texto[:40]}» ha salido demasiado largo ({len(audio) / modelo.sr:.1f} s): se repite con otra semilla")
    if guardar:
        sf.write(SALIDAS / guardar, audio, modelo.sr)
    return audio


def escuchar(audio, titulo=None):
    if titulo:
        display(Markdown(f"**{titulo}** · {len(audio) / modelo.sr:.1f} s"))
    display(Audio(audio, rate=modelo.sr))

In [ ]:
inicio = time.time()
primera = hablar("Hola. Esta frase no la he dicho yo: la ha generado un modelo a partir de quince segundos de mi voz.",
                 guardar="01_primera_frase.wav")

print(f"{len(primera) / modelo.sr:.1f} s de audio generados en {time.time() - inicio:.0f} s")
escuchar(primera, "Tu voz clonada")
escuchar(mi_voz[: int(8 * frecuencia)], "Tu grabación original, para comparar")

Escucha las dos seguidas. Lo normal es que el timbre se parezca mucho y que el ritmo y la entonación sean algo más neutros que los tuyos. Eso último es lo que se ajusta en el paso siguiente.

Sobre la `semilla`: el modelo elige cada sonido con algo de azar, así que la misma frase sale distinta cada vez. Fijar la semilla hace que el resultado se repita. Si una lectura no te convence, prueba con otra semilla antes de tocar nada más: `hablar(texto, semilla=21)`.

## Lo que falla, y cómo lo vigilamos

Estos modelos tienen un fallo característico: a veces **no saben terminar**. Dicen la frase bien y después repiten la última palabra o añaden unos segundos de sonidos sin sentido. Pasa sobre todo en dos casos: con frases muy cortas («Sí.», «¿Se nota?») y con la expresividad muy alta.

Por eso `hablar` hace una comprobación sencilla. Un texto leído en voz alta dura más o menos lo que marca su longitud; si el audio sale mucho más largo, es que el modelo ha añadido algo, y la función lo repite con otra semilla, hasta tres veces. No es infalible: **escucha siempre el resultado entero antes de usarlo.**

---
# 5 · Hacerla expresiva

El modelo tiene tres mandos. Los dos primeros son los que más se notan.

| Mando | Qué controla | Valores |
|---|---|---|
| `exaggeration` | La intensidad emocional | 0,25 lectura plana · 0,5 normal · 0,8 expresiva · de 1 en adelante exagerada, y con más riesgo de que repita palabras o añada ruidos al final |
| `cfg_weight` | Cuánto se ciñe al ritmo de tu grabación | 0,5 normal · 0,3 más pausado y libre · 0 ignora el ritmo de la referencia |
| `temperature` | Cuánto varía de una lectura a otra | 0,8 por defecto · más alto, más imprevisible |

Van en pareja: al subir `exaggeration` el modelo tiende a hablar más deprisa, y se compensa bajando `cfg_weight`. Por eso los estilos de `ESTILOS` mueven los dos a la vez.

La misma frase, en los cuatro estilos:

In [ ]:
frase = "¡No me lo puedo creer! Llevo toda la semana preparando esta clase, y ahora resulta que la voz que estáis oyendo no la he grabado yo."

for estilo, ajustes in ESTILOS.items():
    audio = hablar(frase, estilo=estilo, guardar=f"02_estilo_{estilo}.wav")
    escuchar(audio, f"{estilo} · exaggeration {ajustes['exaggeration']} · cfg_weight {ajustes['cfg_weight']}")

Escúchalas en orden y decide cuál se parece más a cómo lo dirías tú. No hay una medida fiable de «expresividad»: aquí manda el oído.

## El texto también es un mando

El modelo lee lo que hay escrito, signos incluidos. Una parte grande de la expresividad no está en los parámetros, sino en **cómo escribes el texto**:

| Si escribes… | El modelo… |
|---|---|
| Comas y puntos | Hace pausas. Una frase larga sin comas sale atropellada |
| `¿…?` y `¡…!` | Sube la entonación. En español, el signo de apertura ayuda |
| Puntos suspensivos `…` | Deja la frase en el aire |
| Frases cortas | Suena más rotundo. Frases largas, más narrativo |
| Números y siglas | Los lee como puede. Mejor «tres mil euros» que «3.000 €», y «ce ese i ce» si quieres que deletree |

El mismo contenido, escrito para ser leído y escrito para ser oído:

In [ ]:
para_leer = "El modelo Chatterbox (500M de parámetros) clona una voz con 10-20 s de audio y soporta 23 idiomas."
para_oir = "Chatterbox es un modelo pequeño: quinientos millones de parámetros. ¿Y qué necesita para clonar una voz? Quince segundos de audio. Nada más."

escuchar(hablar(para_leer, estilo="expresivo", guardar="03_escrito_para_leer.wav"), "Escrito para leer")
escuchar(hablar(para_oir, estilo="expresivo", guardar="03_escrito_para_oir.wav"), "Escrito para oír")

> **Prueba tú**
> 1. Escribe una frase tuya y escúchala en `"plano"` y en `"teatral"`.
> 2. Coge el estilo `"expresivo"` y cambia solo un mando: `hablar(frase, estilo="expresivo", cfg_weight=0.1)`. ¿Qué cambia?
> 3. Genera la misma frase con tres semillas distintas. ¿Cuánto se diferencian?

In [ ]:
# Prueba tú
mi_frase = "Que ganas tengo de comerme un plato de croquetas"
escuchar(hablar(mi_frase, estilo="expresivo"), "Mi frase")

---
# 6 · Un texto largo, con guion

Para un texto de varias frases hay dos maneras: dárselo entero al modelo, o **ir frase a frase** y unir los audios. La segunda es más trabajo y tiene tres ventajas:

- Si una frase sale mal, se repite solo esa.
- Las pausas entre frases las decides tú.
- **Cada frase puede llevar su propio estilo.** Eso es lo que convierte una lectura correcta en una lectura expresiva: nadie lee un texto entero en el mismo tono.

Un guion es una lista de parejas: la frase y el estilo con que se dice. Una precaución, por lo que vimos en el paso 4: **no dejes sola una frase muy corta.** Júntala con la anterior o con la siguiente.

In [ ]:
GUION = [
    ("Hace un mes, yo no sabía nada de esto.", "natural"),
    ("Hoy he grabado quince segundos de mi voz con el micrófono del portátil, y un modelo de código abierto ha aprendido a imitarla.", "natural"),
    ("¿Asusta un poco? Sí, bastante.", "expresivo"),
    ("Pero también es una herramienta extraordinaria: para quien ha perdido la voz, para doblar un vídeo a otro idioma, para que un texto largo se pueda escuchar.", "expresivo"),
    ("Lo importante es saber cómo funciona, qué puede hacer… y qué no.", "plano"),
    ("¡Y eso es justo lo que vamos a ver hoy, con vuestra propia voz!", "teatral"),
]


def leer_guion(guion, pausa=0.35, guardar=None):
    """Genera cada frase con su estilo y las une, dejando `pausa` segundos de silencio entre una y otra."""
    silencio = np.zeros(int(pausa * modelo.sr), dtype="float32")
    trozos = []
    for n, (texto, estilo) in enumerate(guion, start=1):
        print(f"{n}/{len(guion)} [{estilo}] {texto[:70]}")
        trozos += [hablar(texto, estilo=estilo), silencio]
    audio = np.concatenate(trozos[:-1])
    if guardar:
        sf.write(SALIDAS / guardar, audio, modelo.sr)
    return audio


inicio = time.time()
lectura = leer_guion(GUION, guardar="04_lectura_con_guion.wav")
print(f"\n{len(lectura) / modelo.sr:.0f} s de audio generados en {time.time() - inicio:.0f} s")
escuchar(lectura, "Lectura con guion")

Si tienes un texto seguido y no quieres marcar el estilo frase a frase, esta función lo parte sola por los signos de puntuación, une las frases demasiado cortas y lo lee todo con el mismo estilo.

In [ ]:
def leer(texto, estilo="expresivo", pausa=0.35, guardar=None):
    """Parte un texto en frases y lo lee entero con el mismo estilo."""
    frases = []
    for frase in re.split(r"(?<=[.!?…])\s+", " ".join(texto.split())):
        if frases and len(frases[-1]) < 40:          # una frase muy corta se une a la siguiente
            frases[-1] += " " + frase
        else:
            frases.append(frase)
    if len(frases) > 1 and len(frases[-1]) < 40:     # y si la corta es la última, a la anterior
        frases[-2:] = [frases[-2] + " " + frases[-1]]
    return leer_guion([(frase, estilo) for frase in frases], pausa=pausa, guardar=guardar)


TEXTO = """
Buenos días a todos. Antes de empezar, una advertencia: esta bienvenida no la estoy leyendo yo.
La escribí anoche, y la voz que oís la ha puesto un modelo. ¿Se nota? Decídmelo vosotros al terminar.
"""

escuchar(leer(TEXTO, guardar="05_mi_texto.wav"), "Mi texto")

Los archivos están en la carpeta `audios/`, en formato WAV: se pueden abrir con cualquier reproductor, montar en un vídeo o enviar.

> **Prueba tú**
> 1. Escribe tu propio `GUION`: seis o siete frases para presentarte, o para abrir una clase. Decide el estilo de cada una.
> 2. Cambia `pausa` a 0.8 en una lectura solemne y a 0.15 en una rápida.
> 3. Una frase ha salido rara. Genérala sola con otra semilla hasta que te guste: `hablar(frase, estilo="natural", semilla=3)`.

---
# 7 · Tu voz en otro idioma

El timbre y el idioma van por separado. Con la misma grabación en español, el modelo puede hablar cualquiera de sus 23 idiomas: basta con cambiar `idioma` y escribir el texto en esa lengua.

Hay un efecto secundario: como la referencia está en español, la voz tiende a conservar **acento español**. Si quieres reducirlo, se baja `cfg_weight` a 0, que hace que el modelo ignore el ritmo de la referencia.

In [ ]:
OTROS_IDIOMAS = {
    "en": "Hello! This is my own voice, speaking a language I never recorded.",
    "fr": "Bonjour ! C'est ma propre voix, dans une langue que je n'ai jamais enregistrée.",
    "it": "Ciao! Questa è la mia voce, in una lingua che non ho mai registrato.",
}

for idioma, texto in OTROS_IDIOMAS.items():
    audio = hablar(texto, idioma=idioma, cfg_weight=0.0, guardar=f"06_idioma_{idioma}.wav")
    escuchar(audio, SUPPORTED_LANGUAGES[idioma])

> **Prueba tú**
> 1. Genera la frase en inglés con `cfg_weight=0.5` y con `cfg_weight=0.0`. ¿En cuál se nota más el acento?
> 2. Prueba un idioma que no hables. Si conoces a alguien que lo hable, pregúntale cómo suena.

---
# 8 · ¿Se parece?

A nuestro oído le cuesta juzgar nuestra propia voz: la oímos distinta desde dentro que en una grabación. Conviene tener, además, una medida.

El modelo lleva dentro un **codificador de voz**, la pieza que resume quién habla en un vector. Podemos usarlo nosotros: se calcula el vector de dos audios y se mide cuánto se parecen, con la similitud del coseno. Cerca de 1, la misma voz; más bajo, voces distintas.

Para que el número signifique algo hace falta un punto de comparación. Usamos la **voz de fábrica** del modelo, la que trae guardada y que apartamos en el paso 2, antes de darle tu grabación.

In [ ]:
def huella(audio, frecuencia_audio):
    """Vector que resume quién habla en un audio, calculado con el codificador de voz del modelo."""
    audio_16k = librosa.resample(np.asarray(audio, dtype="float32"), orig_sr=frecuencia_audio, target_sr=16_000)
    vector = np.asarray(modelo.ve.embeds_from_wavs([audio_16k], sample_rate=16_000)).reshape(-1)
    return vector / np.linalg.norm(vector)


tu_voz_en_el_modelo = modelo.conds          # el modelo recuerda la última voz que se le dio: la apartamos un momento
modelo.conds = VOZ_DE_FABRICA
torch.manual_seed(7)
onda = modelo.generate("Y esta es la voz que trae el modelo cuando no se le da ninguna grabación.", language_id="es")
modelo.conds = tu_voz_en_el_modelo
voz_de_fabrica = onda.squeeze(0).cpu().numpy()
escuchar(voz_de_fabrica, "La voz de fábrica del modelo")

referencia = huella(mi_voz, frecuencia)
print("Parecido con tu grabación (1 = la misma voz)\n")
for nombre, audio in [("Tu primera frase clonada", primera), ("La lectura con guion", lectura), ("La voz de fábrica del modelo", voz_de_fabrica)]:
    print(f"  {nombre:32} {float(referencia @ huella(audio, modelo.sr)):.2f}")

Los audios generados con tu voz tienen que quedar claramente por encima de la voz de fábrica. Si quedan cerca, el modelo no ha captado tu timbre, y casi siempre la causa está en la grabación: ruido, eco o muy poca duración.

Esta medida solo mira el **timbre**. No dice si la entonación es natural ni si se entiende bien: eso se sigue juzgando escuchando.

---
# 9 · Marca de agua y uso responsable

## Todo lo que genera el modelo va marcado

Chatterbox añade a cada audio una **marca de agua**: una señal que no se oye y que un detector puede encontrar después, incluso si el archivo se comprime a MP3 o se edita. El detector es público, y es el mismo paquete que instalamos sin saberlo con el modelo.

In [ ]:
import perth

detector = perth.PerthImplicitWatermarker()

clon_guardado, frecuencia_clon = librosa.load(SALIDAS / "01_primera_frase.wav", sr=None)

print("Marca de agua (1 = generado por el modelo, 0 = no)\n")
print("  Tu grabación original :", detector.get_watermark(mi_voz, sample_rate=frecuencia))
print("  La voz clonada        :", detector.get_watermark(clon_guardado, sample_rate=frecuencia_clon))

La marca ayuda, pero no es una garantía: el modelo es de código abierto, y quien quiera puede quitar esa línea del programa. Sirve para detectar audios generados de buena fe, no para frenar a quien quiere engañar.

## Lo que no se hace con esto

- **No se clona la voz de nadie sin su permiso.** Ni la de un compañero «para probar», ni la de una persona conocida. En España la voz es un dato personal y está protegida por el derecho a la propia imagen.
- **Se avisa siempre.** Si enseñas un audio generado, di que lo es. Un audio falso presentado como real es un engaño, aunque la voz sea la tuya.
- **No se usa para hacerse pasar por nadie**: ni llamadas, ni mensajes de voz, ni verificaciones por voz.
- **Tu grabación es tuya.** Con quince segundos de tu voz, cualquiera puede hacer lo que has hecho hoy. Piensa dónde publicas audios tuyos.

Y hay usos que justifican que esto exista: devolver su propia voz a personas que la han perdido por una enfermedad, doblar material docente a otros idiomas con la voz del autor, hacer accesible un texto largo, o grabar la locución de un vídeo sin estudio.

---
# Resumen

| Paso | Lo que hay que recordar |
|---|---|
| Grabación | 10-20 segundos, sin ruido ni eco, hablando como quieres que suene la copia |
| Clonación | No hay entrenamiento: el modelo usa tu audio como referencia en cada frase |
| Expresividad | `exaggeration` arriba y `cfg_weight` abajo; y, sobre todo, escribir el texto para ser oído |
| Textos largos | Frase a frase, cada una con su estilo |
| Otros idiomas | El timbre se conserva; el acento también, salvo con `cfg_weight=0` |
| Comprobación | El parecido de timbre se mide; la naturalidad se escucha |
| Responsabilidad | Solo tu voz o con permiso, avisando siempre de que es un audio generado |

## Para seguir

- **Otra persona, con su permiso.** Cambia `ARCHIVO_DE_VOZ` por su grabación y repite los pasos 4 a 8.
- **Una voz con emociones distintas.** Graba dos referencias, una tranquila y otra enfadada, y compara qué hereda el modelo de cada una.
- **Un audiolibro.** Lee un archivo de texto con `Path("capitulo.txt").read_text()` y pásalo a `leer`.
- **Comprobar lo que dice.** Transcribe el audio generado con Whisper y compáralo con el texto original: así se detectan palabras cambiadas o inventadas.